In [ ]:
import cv2
import numpy as np
import math
from random import randint

class Block:
    """Lớp lưu trữ thông tin của từng ô vuông trên bàn cờ"""
    def __init__(self, i, j):
        self.value = None # None, "x", hoặc "o"
        self.pos = (i, j)

    def set_value(self, value):
        self.value = value

class TicTacToeGUI:
    def __init__(self):
        self.window_name = "TicTacToe AI"
        self.width, self.height = 400, 400
        self.menu_height = 100 # Khu vực hiển thị text bên dưới
        self.image = np.zeros((self.height + self.menu_height, self.width, 3), np.uint8)
        self.turn = 1 # 1 là lượt người (x), -1 là lượt AI (o)
        self.vs_com = 1 # 1: Đánh với máy, 0: 2 người đánh với nhau
        self.reset_game()

    def reset_game(self):
        """Khởi tạo lại bàn cờ trống"""
        self.blocks = []
        self.win = False
        self.change = True
        self.selected = False
        
        # Chia tọa độ 400x400 thành 9 ô vuông nhỏ
        for i in range(3):
            row = []
            for j in range(3):
                pt1 = (j * (self.width // 3) + 3, i * (self.height // 3) + 3) # Góc trên trái
                pt2 = ((j + 1) * (self.width // 3) - 3, (i + 1) * (self.height // 3) - 3) # Góc dưới phải
                row.append([Block(i, j), pt1, pt2])
            self.blocks.append(row)

    def draw_gui(self):
        """Vẽ toàn bộ giao diện (Khung cờ, chữ X/O và Menu)"""
        self.image = np.zeros((self.height + self.menu_height, self.width, 3), np.uint8)
        
        # Vẽ 9 ô cờ
        for i in range(3):
            for j in range(3):
                start_point, end_point = self.blocks[i][j][1], self.blocks[i][j][2]
                cv2.rectangle(self.image, start_point, end_point, (255, 255, 255), -1) # Nền trắng
                
                value = " " if self.blocks[i][j][0].value is None else self.blocks[i][j][0].value
                # Đặt chữ X hoặc O vào giữa ô
                cv2.putText(self.image, value, (j * (self.width // 3) + 25, (i * self.height // 3) + 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 4, (0, 0, 0), 5)

        # Hiển thị trạng thái trò chơi (Thắng/Thua/Hòa/Tới lượt)
        if self.check_win():
            text = "Ban Thang!" if self.turn != self.vs_com else "May Thang!"
        elif self.check_draw():
            text = "Hoa nhau!"
        else:
            text = "Luot cua ban" if self.turn != self.vs_com else "May dang nghi..."
            
        cv2.putText(self.image, text, (self.width // 2 - 80, self.height + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(self.image, "Phim R: Choi lai | Phim ESC: Thoat", (10, self.height + 65), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    def mouse_click(self, event, x, y, flags, param):
        """Bắt sự kiện click chuột từ người chơi"""
        if event == cv2.EVENT_LBUTTONDOWN and not self.win and self.turn != self.vs_com:
            # Kiểm tra xem chuột click vào ô nào
            for i in range(3):
                for j in range(3):
                    pt1, pt2 = self.blocks[i][j][1], self.blocks[i][j][2]
                    if self.blocks[i][j][0].value is None and (pt1[0] <= x <= pt2[0]) and (pt1[1] <= y <= pt2[1]):
                        self.blocks[i][j][0].set_value("x")
                        self.change = True
                        self.selected = True
                        self.turn *= -1 # Chuyển lượt
                        return

    # -------- LOGIC KIỂM TRA THẮNG / HÒA --------
    def check_win(self):
        self.win = False
        b = self.blocks
        # Kiểm tra hàng ngang và hàng dọc
        for i in range(3):
            if b[i][0][0].value and b[i][0][0].value == b[i][1][0].value == b[i][2][0].value: self.win = True
            if b[0][i][0].value and b[0][i][0].value == b[1][i][0].value == b[2][i][0].value: self.win = True
        # Kiểm tra 2 đường chéo
        if b[0][0][0].value and b[0][0][0].value == b[1][1][0].value == b[2][2][0].value: self.win = True
        if b[2][0][0].value and b[2][0][0].value == b[1][1][0].value == b[0][2][0].value: self.win = True
        return self.win

    def check_draw(self):
        for i in range(3):
            for j in range(3):
                if self.blocks[i][j][0].value is None:
                    return False
        return True

    # -------- AI MINIMAX --------
    def next_move_ai(self):
        """Tìm nước đi tốt nhất cho máy (Máy là 'o')"""
        best_score = math.inf # Đang tìm min cho máy
        best_block = None
        
        for i in range(3):
            for j in range(3):
                block = self.blocks[i][j][0]
                if block.value is None:
                    block.value = "o" # Thử đi
                    score = self.minimax(0, True) # Gọi minimax tìm MAX cho người
                    block.value = None # Hoàn tác
                    
                    if score < best_score:
                        best_score = score
                        best_block = block
        return best_block

    def minimax(self, depth, is_maximizing):
        # 1. Đánh giá trạng thái
        if self.check_win():
            return 10 - depth if is_maximizing else -10 + depth # is_maximizing = True tức là máy vừa đánh (Win)
        if self.check_draw():
            return 0
            
        # 2. Duyệt đệ quy
        if is_maximizing: # Lượt của người (tìm Max cho X)
            best_score = -math.inf
            for i in range(3):
                for j in range(3):
                    if self.blocks[i][j][0].value is None:
                        self.blocks[i][j][0].value = "x"
                        score = self.minimax(depth + 1, False)
                        self.blocks[i][j][0].value = None
                        best_score = max(score, best_score)
            return best_score
        else: # Lượt của máy (tìm Min cho O)
            best_score = math.inf
            for i in range(3):
                for j in range(3):
                    if self.blocks[i][j][0].value is None:
                        self.blocks[i][j][0].value = "o"
                        score = self.minimax(depth + 1, True)
                        self.blocks[i][j][0].value = None
                        best_score = min(score, best_score)
            return best_score

    # -------- VÒNG LẶP CHÍNH --------
    def run(self):
        cv2.namedWindow(self.window_name)
        cv2.setMouseCallback(self.window_name, self.mouse_click)
        
        while True:
            if self.change:
                self.draw_gui()
                self.change = False
                
                # Nếu tới lượt máy và game chưa kết thúc
                if self.turn == self.vs_com and not (self.check_win() or self.check_draw()):
                    cv2.imshow(self.window_name, self.image)
                    cv2.waitKey(10) # Dừng 1 chút để render chữ "Máy đang nghĩ"
                    
                    block = self.next_move_ai()
                    if block:
                        block.set_value("o")
                    self.turn *= -1
                    self.change = True
                
            cv2.imshow(self.window_name, self.image)
            key = cv2.waitKey(30)
            
            if key == 27: # Ấn ESC để thoát
                break
            elif key in [ord("r"), ord("R")]: # Ấn R để reset
                self.reset_game()
                self.turn = 1
                
        cv2.destroyAllWindows()

if __name__ == "__main__":
    game = TicTacToeGUI()
    game.run()